# Battery SoC Estimation & Remaining Charging Time Prediction
## Complete Day 1–7 Google Colab Notebook

**Project:** Battery State of Charge (SoC) Estimation & Remaining Charging Time Prediction Using Machine Learning

This notebook prepares and inspects the dataset through **Day 7**.

**Important**
- No ML model training is performed.
- `Power` is derived as `Voltage × Current`.
- The raw CSV files do not contain direct `SoC` or `Remaining Charging Time` columns; target construction must be defined/confirmed before supervised training.
- Raw dataset files are not modified.


## 1. Upload the dataset ZIP

In [1]:
from google.colab import files

uploaded = files.upload()

print("Uploaded file(s):")
for filename in uploaded.keys():
    print(filename)


Saving SOC DATA SET.zip to SOC DATA SET.zip
Uploaded file(s):
SOC DATA SET.zip


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Extract the dataset

In [2]:
import os
import zipfile

zip_file = next(iter(uploaded.keys()))
EXTRACT_DIR = "/content/SOC_DATASET"

os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(zip_file, "r") as zip_ref:
    zip_ref.extractall(EXTRACT_DIR)

print("Dataset extracted successfully.")
print("Extraction directory:", EXTRACT_DIR)


Dataset extracted successfully.
Extraction directory: /content/SOC_DATASET


## 3. Check the folder structure

In [3]:
for root, dirs, files_in_dir in os.walk(EXTRACT_DIR):
    csv_count = sum(f.lower().endswith(".csv") for f in files_in_dir)
    if csv_count:
        print(root, "->", csv_count, "CSV files")


/content/SOC_DATASET/Dataset_Li-ion/25degC -> 36 CSV files
/content/SOC_DATASET/Dataset_Li-ion/40degC -> 35 CSV files
/content/SOC_DATASET/Dataset_Li-ion/n20degC -> 35 CSV files
/content/SOC_DATASET/Dataset_Li-ion/10degC -> 32 CSV files
/content/SOC_DATASET/Dataset_Li-ion/n10degC -> 38 CSV files
/content/SOC_DATASET/Dataset_Li-ion/0degC -> 32 CSV files


## 4. Find all CSV files

In [4]:
import glob

DATASET_DIR = "/content/SOC_DATASET/Dataset_Li-ion"

csv_files = sorted(glob.glob(
    os.path.join(DATASET_DIR, "**", "*.csv"),
    recursive=True
))

print("Total CSV files found:", len(csv_files))

for file in csv_files[:10]:
    print(file)

assert len(csv_files) > 0, "No CSV files found. Check DATASET_DIR."


Total CSV files found: 208
/content/SOC_DATASET/Dataset_Li-ion/0degC/585_C20DisCh.csv
/content/SOC_DATASET/Dataset_Li-ion/0degC/585_Dis_0p5C.csv
/content/SOC_DATASET/Dataset_Li-ion/0degC/585_Dis_2C.csv
/content/SOC_DATASET/Dataset_Li-ion/0degC/585_HPPC.csv
/content/SOC_DATASET/Dataset_Li-ion/0degC/589_Cap_1C.csv
/content/SOC_DATASET/Dataset_Li-ion/0degC/589_Charge1.csv
/content/SOC_DATASET/Dataset_Li-ion/0degC/589_Charge2.csv
/content/SOC_DATASET/Dataset_Li-ion/0degC/589_Charge3.csv
/content/SOC_DATASET/Dataset_Li-ion/0degC/589_Charge4.csv
/content/SOC_DATASET/Dataset_Li-ion/0degC/589_Charge5.csv


## 5. Import libraries and create output folder

In [5]:
import pandas as pd
import numpy as np
import json
import warnings
import shutil

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)

OUTPUT_DIR = "/content/DAY1-7_RESULTS"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Libraries imported.")
print("Output directory:", OUTPUT_DIR)


Libraries imported.
Output directory: /content/DAY1-7_RESULTS


## 6. Create a robust CSV parser

In [6]:
RAW_NUMERIC_COLUMNS = [
    "Voltage", "Current", "Temperature",
    "Capacity", "WhAccu", "Cnt"
]

def find_header_line(file):
    with open(file, "r", encoding="utf-8", errors="ignore") as f:
        for i, line in enumerate(f):
            if (
                "Time Stamp" in line
                and "Voltage" in line
                and "Current" in line
                and "Temperature" in line
            ):
                return i
    return None

def read_battery_csv(file):
    header_line = find_header_line(file)

    if header_line is None:
        raise ValueError("Measurement header not found")

    df = pd.read_csv(
        file,
        skiprows=header_line,
        low_memory=False
    )

    df = df.dropna(how="all").copy()

    # The row immediately after the header is the units row in this dataset.
    # Remove it when it contains unit-like values rather than measurements.
    if len(df) > 0:
        first = df.iloc[0].astype(str).str.strip().str.lower()
        unit_tokens = {
            "sec", "s", "v", "a", "degc", "°c",
            "ah", "wh", "unit", "units"
        }

        unit_hits = sum(
            value in unit_tokens or any(
                token in value for token in unit_tokens
            )
            for value in first.tolist()
        )

        if unit_hits >= 2:
            df = df.iloc[1:].copy()

    for col in RAW_NUMERIC_COLUMNS:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    if "Voltage" in df.columns and "Current" in df.columns:
        df["Power"] = df["Voltage"] * df["Current"]

    return df

print("Parser ready.")


Parser ready.


## 7. Test one CSV

In [7]:
sample_file = csv_files[0]
header_line = find_header_line(sample_file)
sample_df = read_battery_csv(sample_file)

print("Sample file:", sample_file)
print("Header line:", header_line)
print("Rows:", len(sample_df))
print("Columns:", len(sample_df.columns))
print("\nColumns:")
print(sample_df.columns.tolist())
display(sample_df.head())


Sample file: /content/SOC_DATASET/Dataset_Li-ion/0degC/585_C20DisCh.csv
Header line: 28
Rows: 2244
Columns: 16

Columns:
['Time Stamp', 'Step', 'Status', 'Prog Time', 'Step Time', 'Cycle', 'Cycle Level', 'Procedure', 'Voltage', 'Current', 'Temperature', 'Capacity', 'WhAccu', 'Cnt', 'Unnamed: 14', 'Power']


,Time Stamp,Step,Status,Prog Time,Step Time,Cycle,Cycle Level,Procedure,Voltage,Current,Temperature,Capacity,WhAccu,Cnt,Unnamed: 14,Power
1,11/27/2018 8:41:18 PM,22.0,DCH,25:19:08.386,00:01:00.004,0.0,0.0,LG_HG2_NN_Char,4.16273,-0.15325,-0.42063,-0.00253,-0.01052,13.0,NaN,-0.637938
2,11/27/2018 8:42:18 PM,22.0,DCH,25:20:08.381,00:01:59.999,0.0,0.0,LG_HG2_NN_Char,4.15784,-0.15325,-0.42063,-0.00506,-0.02108,13.0,NaN,-0.637189
3,11/27/2018 8:43:18 PM,22.0,DCH,25:21:08.383,00:03:00.001,0.0,0.0,LG_HG2_NN_Char,4.15396,-0.15069,-0.52579,-0.00760,-0.03162,13.0,NaN,-0.625960
4,11/27/2018 8:44:18 PM,22.0,DCH,25:22:08.381,00:03:59.999,0.0,0.0,LG_HG2_NN_Char,4.15042,-0.15325,-0.63095,-0.01014,-0.04216,13.0,NaN,-0.636052
5,11/27/2018 8:45:18 PM,22.0,DCH,25:23:08.382,00:05:00.000,0.0,0.0,LG_HG2_NN_Char,4.14739,-0.15069,-0.31548,-0.01268,-0.05270,13.0,NaN,-0.624970


## Day 1 — Problem Understanding & System Definition

In [8]:
day1 = {
    "project_title": "Battery State of Charge (SoC) Estimation & Remaining Charging Time Prediction Using Machine Learning",
    "problem_statement": (
        "Accurately estimate Battery State of Charge (SoC) and predict the remaining charging time "
        "using measured battery data. The problem is to use measured battery data such as voltage, "
        "current, power, temperature, and charging history to build ML regression models that predict "
        "these two values."
    ),
    "primary_targets": ["Battery State of Charge (SoC)", "Remaining Charging Time"],
    "input_measurements": [
        "Voltage", "Current", "Temperature",
        "Power (derived as Voltage × Current)",
        "Charging history / temporal information"
    ],
    "candidate_models": [
        "Linear Regression", "Decision Tree",
        "Random Forest", "XGBoost",
        "Artificial Neural Network (optional)"
    ],
    "evaluation_metrics": ["MAE", "RMSE", "R²"],
    "important_note": (
        "The raw CSV dataset does not contain direct SoC or Remaining Charging Time columns. "
        "Target construction must therefore be explicitly defined and confirmed before model training."
    )
}

with open(f"{OUTPUT_DIR}/day1_problem_definition.json", "w") as f:
    json.dump(day1, f, indent=4)

print("Day 1 output saved.")


Day 1 output saved.


## Day 2 — Literature Review & Candidate Methods

In [9]:
models_data = pd.DataFrame({
    "Model": [
        "Linear Regression", "Decision Tree", "Random Forest",
        "XGBoost", "Artificial Neural Network"
    ],
    "Role": [
        "Baseline", "Nonlinear baseline", "Main tree ensemble",
        "Boosted tree candidate", "Optional nonlinear model"
    ],
    "Priority": ["Required", "Required", "Required", "Required", "Optional"]
})

display(models_data)
models_data.to_csv(
    f"{OUTPUT_DIR}/day2_candidate_models.csv",
    index=False
)


,Model,Role,Priority
0,Linear Regression,Baseline,Required
1,Decision Tree,Nonlinear baseline,Required
2,Random Forest,Main tree ensemble,Required
3,XGBoost,Boosted tree candidate,Required
4,Artificial Neural Network,Optional nonlinear model,Optional


## Day 3 — Research Framing

In [10]:
day3 = {
    "research_question": (
        "Can machine learning models accurately estimate battery State of Charge and predict "
        "remaining charging time from measured battery operating data across different temperatures "
        "and charging conditions?"
    ),
    "target_1": "Battery State of Charge (SoC)",
    "target_2": "Remaining Charging Time",
    "evaluation": "Compare regression models using MAE, RMSE and R² separately for both targets.",
    "data_split": (
        "Use leakage-safe session-level splitting rather than random row-level splitting "
        "when session identification is finalized."
    )
}

with open(f"{OUTPUT_DIR}/day3_research_question.json", "w") as f:
    json.dump(day3, f, indent=4)

print("Day 3 output saved.")


Day 3 output saved.


## Day 4 — Dataset Design & Quality Expectations

In [11]:
file_inventory_df = pd.DataFrame([
    {
        "File": os.path.basename(file),
        "Full_Path": file,
        "Temperature_Group": os.path.basename(os.path.dirname(file))
    }
    for file in csv_files
])

display(file_inventory_df.head())
file_inventory_df.to_csv(
    f"{OUTPUT_DIR}/file_inventory.csv",
    index=False
)

temperature_summary = (
    file_inventory_df["Temperature_Group"]
    .value_counts()
    .sort_index()
    .rename_axis("Temperature_Group")
    .reset_index(name="File_Count")
)

display(temperature_summary)
temperature_summary.to_csv(
    f"{OUTPUT_DIR}/temperature_file_summary.csv",
    index=False
)


,File,Full_Path,Temperature_Group
0,585_C20DisCh.csv,/content/SOC_DATASET/Dataset_Li-ion/0degC/585_...,0degC
1,585_Dis_0p5C.csv,/content/SOC_DATASET/Dataset_Li-ion/0degC/585_...,0degC
2,585_Dis_2C.csv,/content/SOC_DATASET/Dataset_Li-ion/0degC/585_...,0degC
3,585_HPPC.csv,/content/SOC_DATASET/Dataset_Li-ion/0degC/585_...,0degC
4,589_Cap_1C.csv,/content/SOC_DATASET/Dataset_Li-ion/0degC/589_...,0degC


,Temperature_Group,File_Count
0,0degC,32
1,10degC,32
2,25degC,36
3,40degC,35
4,n10degC,38
5,n20degC,35


In [12]:
schema_rows = []

for file in csv_files:
    try:
        header_line = find_header_line(file)
        df = pd.read_csv(
            file,
            skiprows=header_line,
            nrows=5,
            low_memory=False
        )
        schema_rows.append({
            "File": os.path.basename(file),
            "Header_Line": header_line,
            "Number_of_Columns": len(df.columns),
            "Columns": " | ".join(df.columns.astype(str))
        })
    except Exception as e:
        schema_rows.append({
            "File": os.path.basename(file),
            "Header_Line": None,
            "Number_of_Columns": None,
            "Columns": f"ERROR: {e}"
        })

schema_df = pd.DataFrame(schema_rows)
display(schema_df.head())
schema_df.to_csv(
    f"{OUTPUT_DIR}/schema_inspection.csv",
    index=False
)

data_dictionary = pd.DataFrame({
    "Column": [
        "Time Stamp", "Step", "Status", "Prog Time", "Step Time",
        "Cycle", "Cycle Level", "Procedure", "Voltage", "Current",
        "Temperature", "Capacity", "WhAccu", "Cnt", "Power"
    ],
    "Description": [
        "Measurement timestamp", "Battery test step identifier",
        "Battery/test status", "Program elapsed time", "Step elapsed time",
        "Cycle number", "Cycle level information", "Test procedure",
        "Battery voltage", "Battery current", "Battery temperature",
        "Measured capacity", "Accumulated energy", "Counter/index field",
        "Derived electrical power"
    ],
    "Source": ["Raw"] * 14 + ["Derived"]
})

display(data_dictionary)
data_dictionary.to_csv(
    f"{OUTPUT_DIR}/data_dictionary.csv",
    index=False
)


,File,Header_Line,Number_of_Columns,Columns
0,585_C20DisCh.csv,28,15,Time Stamp | Step | Status | Prog Time | Step ...
1,585_Dis_0p5C.csv,28,15,Time Stamp | Step | Status | Prog Time | Step ...
2,585_Dis_2C.csv,28,15,Time Stamp | Step | Status | Prog Time | Step ...
3,585_HPPC.csv,28,15,Time Stamp | Step | Status | Prog Time | Step ...
4,589_Cap_1C.csv,28,15,Time Stamp | Step | Status | Prog Time | Step ...


,Column,Description,Source
0,Time Stamp,Measurement timestamp,Raw
1,Step,Battery test step identifier,Raw
2,Status,Battery/test status,Raw
3,Prog Time,Program elapsed time,Raw
4,Step Time,Step elapsed time,Raw
5,Cycle,Cycle number,Raw
6,Cycle Level,Cycle level information,Raw
7,Procedure,Test procedure,Raw
8,Voltage,Battery voltage,Raw
9,Current,Battery current,Raw


## Day 5 — Pipeline & Experiment Planning

In [13]:
experiment_plan = pd.DataFrame({
    "Experiment": ["E1", "E2", "E3", "E4", "E5", "E6"],
    "Model": [
        "Linear Regression", "Decision Tree", "Random Forest",
        "XGBoost", "ANN", "Best Model Tuning"
    ],
    "Purpose": [
        "Baseline", "Nonlinear baseline", "Main ensemble",
        "Boosted tree candidate", "Optional nonlinear comparison",
        "Improve shortlisted model"
    ],
    "Status": ["Planned", "Planned", "Planned", "Planned", "Optional", "Future"]
})

display(experiment_plan)
experiment_plan.to_csv(
    f"{OUTPUT_DIR}/experiment_plan.csv",
    index=False
)

experiment_log = pd.DataFrame({
    "Experiment": ["E1", "E2", "E3", "E4", "E5", "E6"],
    "Model": [
        "Linear Regression", "Decision Tree", "Random Forest",
        "XGBoost", "ANN", "Best Model Tuning"
    ],
    "MAE_SoC": [np.nan] * 6,
    "RMSE_SoC": [np.nan] * 6,
    "R2_SoC": [np.nan] * 6,
    "MAE_Remaining_Time": [np.nan] * 6,
    "RMSE_Remaining_Time": [np.nan] * 6,
    "R2_Remaining_Time": [np.nan] * 6
})

experiment_log.to_csv(
    f"{OUTPUT_DIR}/experiment_log_template.csv",
    index=False
)

print("Day 5 outputs saved. Model metrics remain blank until Week 3.")


,Experiment,Model,Purpose,Status
0,E1,Linear Regression,Baseline,Planned
1,E2,Decision Tree,Nonlinear baseline,Planned
2,E3,Random Forest,Main ensemble,Planned
3,E4,XGBoost,Boosted tree candidate,Planned
4,E5,ANN,Optional nonlinear comparison,Optional
5,E6,Best Model Tuning,Improve shortlisted model,Future


Day 5 outputs saved. Model metrics remain blank until Week 3.


## Day 6 — Pre-Dataset Readiness & Methodology

In [14]:
methodology = {
    "workflow": [
        "Dataset", "Data Quality Assessment", "Cleaning", "EDA",
        "Feature Preparation", "Session-Level Split", "Baseline Model",
        "Random Forest / XGBoost", "Evaluation", "Error and Failure Analysis"
    ],
    "current_stage": "Dataset receipt and initial inspection",
    "next_stage": "Data quality assessment and EDA",
    "target_warning": (
        "Raw SoC and Remaining Charging Time columns are not present. "
        "Target construction must be confirmed before supervised model training."
    )
}

with open(f"{OUTPUT_DIR}/day6_methodology.json", "w") as f:
    json.dump(methodology, f, indent=4)

print("Day 6 methodology saved.")


Day 6 methodology saved.


## Day 7 — Dataset Receipt & Initial Inspection

In [15]:
file_summary = []
failed_files = []
total_rows = 0

for index, file in enumerate(csv_files):
    try:
        df = read_battery_csv(file)

        file_summary.append({
            "File": os.path.basename(file),
            "Temperature": os.path.basename(os.path.dirname(file)),
            "Rows": len(df),
            "Columns": len(df.columns),
            "Missing_Cells": int(df.isna().sum().sum()),
            "Duplicate_Rows": int(df.duplicated().sum()),
            "Min_Voltage": df["Voltage"].min() if "Voltage" in df.columns else np.nan,
            "Max_Voltage": df["Voltage"].max() if "Voltage" in df.columns else np.nan,
            "Min_Current": df["Current"].min() if "Current" in df.columns else np.nan,
            "Max_Current": df["Current"].max() if "Current" in df.columns else np.nan,
            "Min_Temperature": df["Temperature"].min() if "Temperature" in df.columns else np.nan,
            "Max_Temperature": df["Temperature"].max() if "Temperature" in df.columns else np.nan
        })

        total_rows += len(df)

    except Exception as e:
        failed_files.append({"File": file, "Error": str(e)})

    if (index + 1) % 20 == 0:
        print(f"Processed {index + 1}/{len(csv_files)} files")

file_summary_df = pd.DataFrame(file_summary)

print("\n====================================")
print("DAY 7 DATASET INSPECTION")
print("====================================")
print("CSV files found       :", len(csv_files))
print("Successfully read     :", len(file_summary_df))
print("Failed files          :", len(failed_files))
print("Total measurement rows:", total_rows)
print("====================================")

file_summary_df.to_csv(
    f"{OUTPUT_DIR}/file_summary.csv",
    index=False
)

temperature_rows = (
    file_summary_df.groupby("Temperature")["Rows"]
    .agg(File_Count="count", Total_Rows="sum")
    .reset_index()
)

display(temperature_rows)
temperature_rows.to_csv(
    f"{OUTPUT_DIR}/temperature_summary.csv",
    index=False
)


Processed 20/208 files
Processed 40/208 files
Processed 60/208 files
Processed 80/208 files
Processed 100/208 files
Processed 120/208 files
Processed 140/208 files
Processed 160/208 files
Processed 180/208 files
Processed 200/208 files

DAY 7 DATASET INSPECTION
CSV files found       : 208
Successfully read     : 208
Failed files          : 0
Total measurement rows: 4954917


,Temperature,File_Count,Total_Rows
0,0degC,32,850622
1,10degC,32,925533
2,25degC,36,1045477
3,40degC,35,692629
4,n10degC,38,829826
5,n20degC,35,610830


### Day 7 — Preliminary test-family summary

In [16]:
def identify_test_type(filename):
    name = filename.lower()
    if "charge" in name:
        return "Charge"
    if "capacity" in name:
        return "Capacity"
    if "discharge" in name:
        return "Discharge"
    if "hppc" in name:
        return "HPPC"
    if "drive" in name or "pause" in name or "mixed" in name:
        return "Drive-cycle/Pause"
    return "Other"

file_summary_df["Test_Type"] = file_summary_df["File"].apply(identify_test_type)

test_summary = (
    file_summary_df.groupby("Test_Type")["Rows"]
    .agg(File_Count="count", Total_Rows="sum")
    .reset_index()
)

display(test_summary)
test_summary.to_csv(
    f"{OUTPUT_DIR}/test_type_summary.csv",
    index=False
)


,Test_Type,File_Count,Total_Rows
0,Charge,98,19336
1,Drive-cycle/Pause,46,2733985
2,HPPC,6,253156
3,Other,58,1948440


### Day 7 — Missing values

In [17]:
missing_records = []

for file in csv_files:
    try:
        df = read_battery_csv(file)
        for column in df.columns:
            count = int(df[column].isna().sum())
            if count > 0:
                missing_records.append({
                    "File": os.path.basename(file),
                    "Column": column,
                    "Missing_Count": count
                })
    except Exception:
        pass

missing_df = pd.DataFrame(missing_records)
display(missing_df)

missing_df.to_csv(
    f"{OUTPUT_DIR}/missing_values_report.csv",
    index=False
)

if len(missing_df):
    overall_missing = (
        missing_df.groupby("Column")["Missing_Count"]
        .sum()
        .reset_index()
        .sort_values("Missing_Count", ascending=False)
    )
else:
    overall_missing = pd.DataFrame(columns=["Column", "Missing_Count"])

display(overall_missing)
overall_missing.to_csv(
    f"{OUTPUT_DIR}/overall_missing_summary.csv",
    index=False
)

print("Total missing cells:", int(overall_missing["Missing_Count"].sum()))


,File,Column,Missing_Count
0,585_C20DisCh.csv,Unnamed: 14,2244
1,585_Dis_0p5C.csv,Unnamed: 14,310
2,585_Dis_2C.csv,Unnamed: 14,230
3,585_HPPC.csv,Unnamed: 14,40219
4,589_Cap_1C.csv,Unnamed: 14,366
...,...,...,...
209,611_Mixed5.csv,Unnamed: 14,45458
210,611_Mixed6.csv,Unnamed: 14,44196
211,611_Mixed7.csv,Unnamed: 14,40683
212,611_Mixed8.csv,Unnamed: 14,46763


,Column,Missing_Count
5,Unnamed: 14,4954060
1,Cnt,4
2,Current,3
0,Capacity,3
3,Power,3
4,Temperature,3
6,Voltage,3
7,WhAccu,3


Total missing cells: 4954082


### Day 7 — Duplicate rows

In [18]:
duplicate_records = []
total_duplicate_rows = 0

for file in csv_files:
    try:
        df = read_battery_csv(file)
        count = int(df.duplicated(keep="first").sum())

        if count > 0:
            duplicate_records.append({
                "File": os.path.basename(file),
                "Duplicate_Rows": count
            })
            total_duplicate_rows += count
    except Exception:
        pass

duplicate_df = pd.DataFrame(duplicate_records)
display(duplicate_df)

duplicate_df.to_csv(
    f"{OUTPUT_DIR}/duplicate_report.csv",
    index=False
)

print("Files containing duplicates:", len(duplicate_df))
print("Duplicate rows beyond first:", total_duplicate_rows)


,File,Duplicate_Rows
0,585_C20DisCh.csv,1
1,585_Dis_0p5C.csv,2
2,585_Dis_2C.csv,2
3,585_HPPC.csv,188
4,589_Cap_1C.csv,2
...,...,...
170,611_Mixed4.csv,1
171,611_Mixed5.csv,1
172,611_Mixed6.csv,1
173,611_Mixed7.csv,1


Files containing duplicates: 175
Duplicate rows beyond first: 5876


### Day 7 — Numerical ranges

In [19]:
range_records = []

for file in csv_files:
    try:
        df = read_battery_csv(file)

        for column in [
            "Voltage", "Current", "Temperature",
            "Capacity", "WhAccu", "Cnt", "Power"
        ]:
            if column in df.columns:
                range_records.append({
                    "File": os.path.basename(file),
                    "Column": column,
                    "Minimum": df[column].min(),
                    "Maximum": df[column].max()
                })
    except Exception:
        pass

range_df = pd.DataFrame(range_records)
display(range_df.head(20))

range_df.to_csv(
    f"{OUTPUT_DIR}/numerical_ranges.csv",
    index=False
)

overall_ranges = []
for column in [
    "Voltage", "Current", "Temperature",
    "Capacity", "WhAccu", "Cnt", "Power"
]:
    subset = range_df[range_df["Column"] == column]
    if len(subset):
        overall_ranges.append({
            "Column": column,
            "Global_Minimum": subset["Minimum"].min(),
            "Global_Maximum": subset["Maximum"].max()
        })

overall_ranges_df = pd.DataFrame(overall_ranges)
display(overall_ranges_df)
overall_ranges_df.to_csv(
    f"{OUTPUT_DIR}/overall_numerical_ranges.csv",
    index=False
)


,File,Column,Minimum,Maximum
0,585_C20DisCh.csv,Voltage,2.799930,4.199950
1,585_C20DisCh.csv,Current,-0.153250,0.150690
2,585_C20DisCh.csv,Temperature,-3.365070,1.261900
3,585_C20DisCh.csv,Capacity,-2.598620,0.088250
4,585_C20DisCh.csv,WhAccu,-9.674280,0.532270
5,585_C20DisCh.csv,Cnt,13.000000,13.000000
6,585_C20DisCh.csv,Power,-0.637938,0.631748
7,585_Dis_0p5C.csv,Voltage,2.799930,4.199950
8,585_Dis_0p5C.csv,Current,-1.499250,3.001080
9,585_Dis_0p5C.csv,Temperature,-0.736110,25.027730


,Column,Global_Minimum,Global_Maximum
0,Voltage,0.000000,4.239900
1,Current,-18.098280,6.004720
2,Temperature,-22.503930,41.327300
3,Capacity,-2.780740,2.818040
4,WhAccu,-10.302800,10.952510
5,Cnt,1.000000,51.000000
6,Power,-69.172028,25.193022


### Day 7 — Domain-range flags

These are inspection flags only and are not automatic outlier-removal rules.

In [20]:
domain_flags = []

for file in csv_files:
    try:
        df = read_battery_csv(file)

        domain_flags.append({
            "File": os.path.basename(file),
            "Voltage_Below_2.8V": int((df["Voltage"] < 2.8).sum()) if "Voltage" in df else 0,
            "Voltage_Above_4.2V": int((df["Voltage"] > 4.2).sum()) if "Voltage" in df else 0,
            "Current_Below_-6A": int((df["Current"] < -6).sum()) if "Current" in df else 0,
            "Current_Above_3A": int((df["Current"] > 3).sum()) if "Current" in df else 0,
            "Temperature_Below_-20C": int((df["Temperature"] < -20).sum()) if "Temperature" in df else 0,
            "Temperature_Above_40C": int((df["Temperature"] > 40).sum()) if "Temperature" in df else 0
        })
    except Exception:
        pass

domain_flags_df = pd.DataFrame(domain_flags)
display(domain_flags_df.head())

domain_flags_df.to_csv(
    f"{OUTPUT_DIR}/domain_flags.csv",
    index=False
)

domain_totals = pd.DataFrame({
    "Flag": [
        "Voltage below 2.8 V", "Voltage above 4.2 V",
        "Current below -6 A", "Current above 3 A",
        "Temperature below -20 °C", "Temperature above 40 °C"
    ],
    "Rows": [
        domain_flags_df["Voltage_Below_2.8V"].sum(),
        domain_flags_df["Voltage_Above_4.2V"].sum(),
        domain_flags_df["Current_Below_-6A"].sum(),
        domain_flags_df["Current_Above_3A"].sum(),
        domain_flags_df["Temperature_Below_-20C"].sum(),
        domain_flags_df["Temperature_Above_40C"].sum()
    ]
})

display(domain_totals)
domain_totals.to_csv(
    f"{OUTPUT_DIR}/domain_flag_summary.csv",
    index=False
)


,File,Voltage_Below_2.8V,Voltage_Above_4.2V,Current_Below_-6A,Current_Above_3A,Temperature_Below_-20C,Temperature_Above_40C
0,585_C20DisCh.csv,1,0,0,0,0,0
1,585_Dis_0p5C.csv,1,0,0,1,0,0
2,585_Dis_2C.csv,1,0,0,0,0,0
3,585_HPPC.csv,6,4,1677,53,0,0
4,589_Cap_1C.csv,2,0,0,0,0,0


,Flag,Rows
0,Voltage below 2.8 V,13023
1,Voltage above 4.2 V,1343
2,Current below -6 A,156305
3,Current above 3 A,96922
4,Temperature below -20 °C,3441
5,Temperature above 40 °C,5988


### Day 7 — Representative files

In [21]:
charge_candidates = [f for f in csv_files if "549_Charge.csv" in f]
charge_file = charge_candidates[0] if charge_candidates else next(
    (f for f in csv_files if "charge" in os.path.basename(f).lower()), None
)

print("Representative charge file:", charge_file)

if charge_file:
    charge_df = read_battery_csv(charge_file)
    print("Rows:", len(charge_df))
    print("Columns:", len(charge_df.columns))
    display(charge_df.head(10))

    with open(f"{OUTPUT_DIR}/representative_charge_file.json", "w") as f:
        json.dump({
            "File": os.path.basename(charge_file),
            "Path": charge_file,
            "Rows": len(charge_df),
            "Columns": len(charge_df.columns),
            "Columns_List": charge_df.columns.tolist()
        }, f, indent=4)

missing_candidates = [f for f in csv_files if "582_LA92.csv" in f]
missing_file = missing_candidates[0] if missing_candidates else None

print("\nRepresentative missing-value file:", missing_file)

if missing_file:
    missing_sample_df = read_battery_csv(missing_file)
    print("Rows:", len(missing_sample_df))
    display(missing_sample_df.isna().sum())


Representative charge file: /content/SOC_DATASET/Dataset_Li-ion/25degC/549_Charge.csv
Rows: 857
Columns: 15


,Time Stamp,Step,Status,Prog Time,Step Time,Cycle,Cycle Level,Procedure,Voltage,Current,Temperature,Capacity,WhAccu,Cnt,Power
1,10/25/2018 4:09:59 AM,4.0,PAU,01:00.0,01:00.0,0.0,0.0,NN_Char_Charge,3.10125,0.0,23.97615,0.0,0.0,11,0.0
2,10/25/2018 4:10:59 AM,4.0,PAU,02:00.0,02:00.0,0.0,0.0,NN_Char_Charge,3.10192,0.0,23.87099,0.0,0.0,11,0.0
3,10/25/2018 4:11:59 AM,4.0,PAU,03:00.0,03:00.0,0.0,0.0,NN_Char_Charge,3.10260,0.0,23.97615,0.0,0.0,11,0.0
4,10/25/2018 4:12:59 AM,4.0,PAU,04:00.0,04:00.0,0.0,0.0,NN_Char_Charge,3.10344,0.0,23.87099,0.0,0.0,11,0.0
5,10/25/2018 4:13:59 AM,4.0,PAU,05:00.0,05:00.0,0.0,0.0,NN_Char_Charge,3.10412,0.0,23.97615,0.0,0.0,11,0.0
6,10/25/2018 4:14:59 AM,4.0,PAU,06:00.0,06:00.0,0.0,0.0,NN_Char_Charge,3.10462,0.0,23.97615,0.0,0.0,11,0.0
7,10/25/2018 4:15:59 AM,4.0,PAU,07:00.0,07:00.0,0.0,0.0,NN_Char_Charge,3.10530,0.0,23.97615,0.0,0.0,11,0.0
8,10/25/2018 4:16:59 AM,4.0,PAU,08:00.0,08:00.0,0.0,0.0,NN_Char_Charge,3.10597,0.0,23.97615,0.0,0.0,11,0.0
9,10/25/2018 4:17:59 AM,4.0,PAU,09:00.0,09:00.0,0.0,0.0,NN_Char_Charge,3.10648,0.0,23.87099,0.0,0.0,11,0.0
10,10/25/2018 4:18:59 AM,4.0,PAU,10:00.0,10:00.0,0.0,0.0,NN_Char_Charge,3.10715,0.0,23.97615,0.0,0.0,11,0.0



Representative missing-value file: /content/SOC_DATASET/Dataset_Li-ion/10degC/582_LA92.csv
Rows: 90740


,0
Time Stamp,0
Step,0
Status,0
Prog Time,0
Step Time,0
Cycle,0
Cycle Level,0
Procedure,0
Voltage,3
Current,3


### Day 7 — Timestamp and Procedure inspection

In [22]:
timestamp_results = []
procedure_records = []

for file in csv_files:
    try:
        df = read_battery_csv(file)

        if "Time Stamp" in df.columns:
            counts = df["Time Stamp"].value_counts()
            timestamp_results.append({
                "File": os.path.basename(file),
                "Unique_Timestamps": int(df["Time Stamp"].nunique()),
                "Repeated_Timestamp_Values": int((counts > 1).sum())
            })

        if "Procedure" in df.columns:
            counts = df["Procedure"].astype(str).value_counts()
            for procedure, count in counts.items():
                procedure_records.append({
                    "File": os.path.basename(file),
                    "Procedure": procedure,
                    "Rows": int(count)
                })

    except Exception:
        pass

timestamp_df = pd.DataFrame(timestamp_results)
procedure_df = pd.DataFrame(procedure_records)

display(timestamp_df.head())
display(procedure_df.head(30))

timestamp_df.to_csv(
    f"{OUTPUT_DIR}/timestamp_inspection.csv",
    index=False
)
procedure_df.to_csv(
    f"{OUTPUT_DIR}/procedure_values.csv",
    index=False
)


,File,Unique_Timestamps,Repeated_Timestamp_Values
0,585_C20DisCh.csv,2242,2
1,585_Dis_0p5C.csv,307,3
2,585_Dis_2C.csv,226,3
3,585_HPPC.csv,11510,3260
4,589_Cap_1C.csv,364,2


,File,Procedure,Rows
0,585_C20DisCh.csv,LG_HG2_NN_Char,2244
1,585_Dis_0p5C.csv,NN_Char_Charge,209
2,585_Dis_0p5C.csv,LG_HG2_NN_Char,101
3,585_Dis_2C.csv,NN_Char_Charge,205
4,585_Dis_2C.csv,LG_HG2_NN_Char,25
5,585_HPPC.csv,HPPC_4pulse,36023
6,585_HPPC.csv,HPPC_Multi_Pulse,3985
7,585_HPPC.csv,NN_Char_Charge,210
8,585_HPPC.csv,LG_HG2_NN_Char,1
9,589_Cap_1C.csv,LG_HG2_CyclesA,366


### Day 7 — Raw target-column check

In [23]:
all_columns = set()

for file in csv_files[:10]:
    try:
        df = read_battery_csv(file)
        all_columns.update(df.columns.tolist())
    except Exception:
        pass

print("Columns found:")
for column in sorted(all_columns):
    print("-", column)

soc_columns = [
    c for c in all_columns
    if "soc" in c.lower()
]

remaining_time_columns = [
    c for c in all_columns
    if (
        "remaining" in c.lower()
        or "charge time" in c.lower()
        or "time remaining" in c.lower()
    )
]

print("\nPossible SoC columns:", soc_columns)
print("Possible remaining-time columns:", remaining_time_columns)


Columns found:
- Capacity
- Cnt
- Current
- Cycle
- Cycle Level
- Power
- Procedure
- Prog Time
- Status
- Step
- Step Time
- Temperature
- Time Stamp
- Unnamed: 14
- Voltage
- WhAccu

Possible SoC columns: []
Possible remaining-time columns: []


### Day 7 — Final report

In [24]:
final_report = {
    "project": "Battery SoC Estimation & Remaining Charging Time Prediction Using Machine Learning",
    "dataset_path": DATASET_DIR,
    "total_csv_files": len(csv_files),
    "successfully_read_files": len(file_summary_df),
    "failed_files": len(failed_files),
    "total_measurement_rows": int(total_rows),
    "temperature_groups": sorted(file_summary_df["Temperature"].unique().tolist()),
    "total_missing_cells": int(overall_missing["Missing_Count"].sum()),
    "files_with_duplicates": len(duplicate_df),
    "duplicate_rows_beyond_first": int(total_duplicate_rows),
    "raw_columns": [
        "Time Stamp", "Step", "Status", "Prog Time", "Step Time",
        "Cycle", "Cycle Level", "Procedure", "Voltage", "Current",
        "Temperature", "Capacity", "WhAccu", "Cnt"
    ],
    "derived_columns": ["Power"],
    "raw_soc_column_present": len(soc_columns) > 0,
    "raw_remaining_time_column_present": len(remaining_time_columns) > 0,
    "power_derivation": "Power = Voltage × Current",
    "target_status": (
        "SoC and Remaining Charging Time targets require explicit construction/"
        "confirmation before supervised model training."
    ),
    "current_stage": "Day 7 - Dataset Receipt and Initial Inspection",
    "next_stage": "Day 8 - Data Quality Assessment"
}

with open(f"{OUTPUT_DIR}/day7_initial_inspection_report.json", "w") as f:
    json.dump(final_report, f, indent=4)

day7_summary = pd.DataFrame({
    "Metric": [
        "Total CSV files", "Successfully read files", "Failed files",
        "Total measurement rows", "Temperature groups", "Total missing cells",
        "Files with duplicate rows", "Duplicate rows beyond first",
        "Raw SoC column present",
        "Raw Remaining Charging Time column present",
        "Power column"
    ],
    "Value": [
        len(csv_files), len(file_summary_df), len(failed_files),
        total_rows, ", ".join(sorted(file_summary_df["Temperature"].unique())),
        int(overall_missing["Missing_Count"].sum()),
        len(duplicate_df), int(total_duplicate_rows),
        len(soc_columns) > 0, len(remaining_time_columns) > 0,
        "Derived: Voltage × Current"
    ]
})

display(day7_summary)

day7_summary.to_csv(
    f"{OUTPUT_DIR}/Day7_Initial_Inspection_Summary.csv",
    index=False
)

print("\nFINAL DAY 7 REPORT")
print("===================")
for k, v in final_report.items():
    print(f"{k}: {v}")


,Metric,Value
0,Total CSV files,208
1,Successfully read files,208
2,Failed files,0
3,Total measurement rows,4954917
4,Temperature groups,"0degC, 10degC, 25degC, 40degC, n10degC, n20degC"
5,Total missing cells,4954082
6,Files with duplicate rows,175
7,Duplicate rows beyond first,5876
8,Raw SoC column present,False
9,Raw Remaining Charging Time column present,False



FINAL DAY 7 REPORT
project: Battery SoC Estimation & Remaining Charging Time Prediction Using Machine Learning
dataset_path: /content/SOC_DATASET/Dataset_Li-ion
total_csv_files: 208
successfully_read_files: 208
failed_files: 0
total_measurement_rows: 4954917
temperature_groups: ['0degC', '10degC', '25degC', '40degC', 'n10degC', 'n20degC']
total_missing_cells: 4954082
files_with_duplicates: 175
duplicate_rows_beyond_first: 5876
raw_columns: ['Time Stamp', 'Step', 'Status', 'Prog Time', 'Step Time', 'Cycle', 'Cycle Level', 'Procedure', 'Voltage', 'Current', 'Temperature', 'Capacity', 'WhAccu', 'Cnt']
derived_columns: ['Power']
raw_soc_column_present: False
raw_remaining_time_column_present: False
power_derivation: Power = Voltage × Current
target_status: SoC and Remaining Charging Time targets require explicit construction/confirmation before supervised model training.
current_stage: Day 7 - Dataset Receipt and Initial Inspection
next_stage: Day 8 - Data Quality Assessment


## Export all Day 1–7 results

In [25]:
RESULTS_ZIP = "/content/DAY1-7_RESULTS.zip"

shutil.make_archive(
    "/content/DAY1-7_RESULTS",
    "zip",
    OUTPUT_DIR
)

print("Results ZIP created:")
print(RESULTS_ZIP)

print("\nGenerated files:")
for root, dirs, files_in_dir in os.walk(OUTPUT_DIR):
    for file in sorted(files_in_dir):
        print(os.path.relpath(
            os.path.join(root, file),
            OUTPUT_DIR
        ))


Results ZIP created:
/content/DAY1-7_RESULTS.zip

Generated files:
Day7_Initial_Inspection_Summary.csv
data_dictionary.csv
day1_problem_definition.json
day2_candidate_models.csv
day3_research_question.json
day6_methodology.json
day7_initial_inspection_report.json
domain_flag_summary.csv
domain_flags.csv
duplicate_report.csv
experiment_log_template.csv
experiment_plan.csv
file_inventory.csv
file_summary.csv
missing_values_report.csv
numerical_ranges.csv
overall_missing_summary.csv
overall_numerical_ranges.csv
procedure_values.csv
representative_charge_file.json
schema_inspection.csv
temperature_file_summary.csv
temperature_summary.csv
test_type_summary.csv
timestamp_inspection.csv
